[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SeyhaLite/ml-foundations/blob/main/docs/notebooks/05_smoke_training.ipynb)

# ០៥ · Smoke Training — ហ្វឹកសាកល្បងដំបូង

In [ ]:
# ⚙️ Setup — រត់ cell នេះមុនគេ (Auto-setup for Google Colab)
import os, subprocess, sys

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir("ml-foundations"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/SeyhaLite/ml-foundations.git"], check=True)
    os.chdir("ml-foundations")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# ធ្វើឱ្យ qwen_ft import បានភ្លាមៗក្នុង session នេះ (មិនបាច់ restart)
for _cand in ("src", os.path.join("..", "..", "src")):
    _cand = os.path.abspath(_cand)
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import qwen_ft
print("qwen-ft", qwen_ft.__version__, "ready ✅")

## គោលដៅ

រត់ការហ្វឹក LoRA **ពិត** លើ dataset ខ្មែរគំរូ — ពី validation ដល់ adapter រក្សាទុក។

```
⚠️ ត្រូវការ: GPU ≥16GB (T4 គ្រប់គ្រាន់សម្រាប់ 0.6B/0.8B) + download ~2GB
```

## ជំហាន ១ — Validate config (រត់បានគ្រប់ម៉ាស៊ីន)

In [ ]:
import os
from pathlib import Path

cfg_path = Path("configs/qwen3-0.6b-lora.yaml")   # ម៉ូដែលតូចបំផុតសម្រាប់ Colab
assert cfg_path.exists(), "រត់ notebook នេះពី repo root!"

from qwen_ft.config import load_config
from qwen_ft.data import validate_jsonl_file

cfg = load_config(cfg_path)
report = validate_jsonl_file(cfg.training.dataset_path)
print(f"✅ config + dataset ({report['records']} records) valid")
print(f"   model={cfg.model.model_id} · r={cfg.lora.r} · epochs={cfg.training.num_epochs}")

steps_per_epoch = max(1, report["records"] // cfg.training.effective_batch_size())
print(f"   optimizer steps/epoch ≈ {steps_per_epoch} (dataset តូច — smoke test គឺគ្រប់គ្រាន់)")

## ជំហាន ២ — ហ្វឹកពិត (guarded)

```bash
pip install -e ".[train]"
export QWEN_FT_ALLOW_HEAVY=1
```

Cell នេះហ្វឹក Qwen3-0.6B លើ 24 ឧទាហរណ៍ — ប៉ុន្មាននាទីលើ T4។
លទ្ធផលមិន «ឆ្លាត» ភ្លាមៗទេ (dataset តូចណាស់) — គោលដៅគឺ **ដំណើរការ**
ដំណើរការ end-to-end៖ loss ធ្លាក់ + adapter រក្សាទុក។

In [ ]:
import os
ALLOW_HEAVY = os.environ.get("QWEN_FT_ALLOW_HEAVY", "0") == "1"
if ALLOW_HEAVY:
    # ហ្វឹក LoRA ពិតលើ Qwen3-0.6B + dataset ខ្មែរគំរូ
    from qwen_ft.training import run_training

    cfg.training.num_epochs = 1          # smoke test — ខ្លីបំផុត
    cfg.training.logging_steps = 1
    cfg.training.save_steps = 1000
    adapter_dir = run_training(cfg)
    print("🎉 Adapter saved to:", adapter_dir)
    print(sorted(os.listdir(adapter_dir)))
else:
    print("⏭️  Heavy demo skipped — set QWEN_FT_ALLOW_HEAVY=1 (requires torch/transformers + model download).")

## ជំហាន ៣ — ផ្ទៀងផ្ទាត់ adapter (guarded)

In [ ]:
import os
ALLOW_HEAVY = os.environ.get("QWEN_FT_ALLOW_HEAVY", "0") == "1"
if ALLOW_HEAVY:
    # ផ្ទុក adapter ហើយសន្ទនា
    from qwen_ft.inference import load_for_chat, chat

    model, tokenizer = load_for_chat(cfg.model.model_id, adapter_path=str(adapter_dir))
    reply = chat(model, tokenizer,
                 [{"role": "user", "content": "តើ LoRA គឺជាអ្វី?"}],
                 max_new_tokens=128, enable_thinking=False)
    print("ចម្លើយ:", reply)
else:
    print("⏭️  Heavy demo skipped — set QWEN_FT_ALLOW_HEAVY=1 (requires torch/transformers + model download).")

## ជម្រើសគ្មាន GPU — offline smoke test

Test suite មាន **smoke test ពិត** ដែលមិន download អ្វីសោះ
(ម៉ូដែល random តូច Qwen3-architecture)៖

```bash
pytest -m heavy -q        # ត្រូវការតែ torch/transformers/peft/trl (CPU គ្រប់គ្រាន់)
```

### លំហាត់
1. ប្តូរ config ទៅ Qwen3.5-0.8B — តើត្រូវការ transformers កំណែប៉ុន្មាន?
2. ដំឡើង epochs ទៅ ៥ — សង្កេត loss (overfit?)
3. អានជំពូក [៦ · ហ្វឹក SFT](../chapters/ch06-training-sft.md)